In [12]:
# %pip install ipynb
import numpy as np
import os
from typing import List, Tuple
from ipynb.fs.full.audio_parser import audio_convert, spectrogram_conversion

def generate_fingerprints(peaks: List[Tuple[int, int]], fanout: int = 5, min_time_delta: int = 0, 
max_time_delta: int = 200) -> List[Tuple[Tuple[int, int, int], int]]:
    
    fingerprints = [] # will return this and add to database
    num_peaks = len(peaks)
    
    for i in range(num_peaks): # iterate through all peaks, call the current iteration's peak as "anchor"
        anchor_time, anchor_freq = peaks[i] # grab time point and freq of peak i, it will act as anchor of fanout
        links_formed = 0
        
        for j in range(i + 1, num_peaks): # iterate thru all peaks AHEAD in time of anchor (no looking backwards), 
                                          # call the current iteration's peak as "j"
            j_time, j_freq = peaks[j] # take time point and freq of j
            time_delta = j_time - anchor_time # calculate time difference between anchor and  j
            if time_delta < min_time_delta: # if anchor and j are too close in time together, they may overlap
                continue                    # let's not include it in fanout
            if time_delta > max_time_delta: # this means anchor and j are too far apart
                break                       # let's not include it in fanout
            if links_formed >= fanout:      # if we have already gotten 5 fanout peaks
                break                       # we can end the j loop
                
            fingerprint_hash = (anchor_freq, j_freq, time_delta) # j is close in time to anchor, so it's local 
            fingerprints.append((fingerprint_hash, anchor_time)) # add anchor_freq, j_freq, and time diff to fanout
            links_formed += 1 # update our fanout counter so we can keep track of when we reach 5 peaks in fanout
            
    return fingerprints # return our list that contains a 5-peak fanout for every peak given 
                        # (i.e. for every peak, its 5 closest peaks AHEAD in time)
    


song_path = os.path.join("Music", "Lover Girl.wav") 
print(f"Processing: {song_path}...")

samples, sample_rate = audio_convert(song_path)

print("Samples shape:", samples.shape)
print("Sample rate:", sample_rate)
print("Max volume sample value:", np.max(np.abs(samples)))

log_spectro, extracted_peaks = spectrogram_conversion(samples, sample_rate)
print("Spectrogram shape:", log_spectro.shape)
print("Min/Max values in Spectrogram:", np.min(log_spectro), np.max(log_spectro))
print(f"Jesse's function extracted {len(extracted_peaks)} peaks.")

my_fingerprints = generate_fingerprints(extracted_peaks, fanout=3)
print(f"Successfully generated {len(my_fingerprints)} unique fingerprints!")

Processing: Music/Lover Girl.wav...
Samples shape: (2635186,)
Sample rate: 16000
Max volume sample value: 0.0077819824
Spectrogram shape: (1025, 2572)
Min/Max values in Spectrogram: -100.0 -67.043594
Jesse's function extracted 0 peaks.
Successfully generated 0 unique fingerprints!
